# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My lane as an ML task**

My lane is a ranking/scoring task because the main decision is to determine which pages should be reviewed first. The system would assign each page a priority score and rank the pages from higher to lower priority. The SEO team could then focus its limited review capacity on the highest-ranked pages rather than treating every page equally.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Walk up the directory tree until data/raw is found
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Unique content items:", df["content_id"].nunique())

Dataset shape: (30000, 44)
Unique content items: 30000


In [16]:
ranking_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_direction",
    "trend_pct"
]

print("Ranking-relevant columns:")
print(ranking_columns)

Ranking-relevant columns:
['content_id', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The intended target is an observed future outcome indicating that a page later becomes a meaningful review opportunity, such as a sustained decline in performance after the feature window. This future outcome would provide a stronger target for ranking pages because it is measured after the decision point. In the starter dataset, a provisional proxy is trend_direction == "down", which indicates current observed decline. I will treat this as a proxy rather than the ideal final target because it is derived from the current performance window rather than a future outcome.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_proxy"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Proxy distribution:")
print(df["is_declining_proxy"].value_counts())

print("\nProxy decline rate:",
      round(df["is_declining_proxy"].mean(), 3))

Proxy distribution:
is_declining_proxy
1    16262
0    13738
Name: count, dtype: int64

Proxy decline rate: 0.542


In [18]:
df[
    [
        "content_id",
        "impressions_90d",
        "trend_direction",
        "trend_pct",
        "is_declining_proxy"
    ]
].head(10)

,content_id,impressions_90d,trend_direction,trend_pct,is_declining_proxy
0,content_304f48230142,3803,down,-41.4,1
1,content_a1fb4e703a9e,15320,down,-57.7,1
2,content_9aa793d4d895,12581,down,-60.9,1
3,content_331d6c4de07b,11751,stable,-13.8,0
4,content_d99b7a2d90ca,19140,down,-34.7,1
5,content_d4084a4bc775,3970,down,-38.9,1
6,content_9a34b442b552,20,down,-92.3,1
7,content_a63219c6e95a,1724,stable,0.6,0
8,content_5e6c160719bc,32574,down,-58.8,1
9,content_c27558df2b0c,1240,down,-29.2,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric will be **Precision@K**, with K chosen according to the number of pages the SEO team can realistically review. For example, Precision@20 measures how many of the top 20 ranked pages match the defined relevance criterion. A higher Precision@K means that the review queue places more useful candidates near the top. The final value of K should therefore reflect the team's actual review capacity.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

baseline_scores = (
    -df["trend_pct"]
)

for k in [20, 50]:
    score = precision_at_k(
        baseline_scores.to_numpy(),
        df["is_declining_proxy"],
        k
    )

    print(f"Precision@{k}: {score:.3f}")

Precision@20: 1.000
Precision@50: 1.000


Above is Demonstration only: Precision@K calculation using the
existing decline signal. This is not the final ML baseline.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one page/content item. Each row represents one pseudonymized content item for a client and contains its search performance, engagement, content, freshness, and trend information. The model or scoring system will therefore produce one priority score for each page, which will then be used to create the ranked review queue.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(
    "One row per content item:",
    len(df) == df["content_id"].nunique()
)

One row per content item: True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could prioritize pages using a small number of thresholds, such as selecting every page whose performance has declined by more than a chosen percentage. However, review priority can depend on several signals at the same time, including visibility, CTR, position, content age, freshness, engagement, search demand, and content characteristics. These signals may interact in ways that are difficult to capture with a few fixed thresholds. Therefore, ML is worth testing against a transparent rule-based baseline to determine whether it can produce a more useful top-ranked review queue. If a simple rule performs equally well, there is no reason to prefer the more complex ML approach.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
simple_rule = (
    (df["trend_direction"].str.lower() == "down") &
    (df["impressions_90d"] > 500)
)

print("Pages selected by the fixed rule:",
      simple_rule.sum())

print("Percentage selected:",
      round(simple_rule.mean() * 100, 2), "%")

Pages selected by the fixed rule: 9956
Percentage selected: 33.19 %


In [22]:
df.loc[
    simple_rule,
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "trend_pct"
    ]
].head(10)

,content_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,trend_pct
0,content_304f48230142,3803,29,0.76,10.6,187,-41.4
1,content_a1fb4e703a9e,15320,7,0.05,20.3,445,-57.7
2,content_9aa793d4d895,12581,11,0.09,36.5,141,-60.9
4,content_d99b7a2d90ca,19140,24,0.13,44.0,263,-34.7
5,content_d4084a4bc775,3970,1,0.03,8.5,147,-38.9
8,content_5e6c160719bc,32574,29,0.09,46.0,90,-58.8
9,content_c27558df2b0c,1240,2,0.16,4.9,257,-29.2
16,content_78bd1d4a1d4d,13848,21,0.15,8.9,307,-39.1
17,content_761a44afda12,9449,7,0.07,7.3,421,-20.2
18,content_0b360eb9db55,5141,7,0.14,11.4,119,-56.9


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.